# Air Pollution Project


## Load Data and Libraries


In [ ]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

import matplotlib.pyplot as plt
from sklearn.impute import SimpleImputer

# Import airpollution data
df_ap = pd.read_csv("data/Train.csv")
df_test = pd.read_csv("data/Test.csv")

df_ap.info()
df_ap.head()


In [ ]:
#visualizing the target over time
df_ap['Date'] = pd.to_datetime(df_ap['Date'])

target_by_date = df_ap.groupby('Date')['target'].mean()

plt.figure(figsize=(12, 5))
plt.plot(target_by_date.index, target_by_date.values)
plt.title('Target over time')
plt.xlabel('Date')
plt.ylabel('Mean target')
plt.grid(True)
plt.show()
place_id = '010Q650'

df_place = df_ap[df_ap['Place_ID'] == place_id].copy()
df_place['Date'] = pd.to_datetime(df_place['Date'])

plt.figure(figsize=(12, 5))
plt.plot(df_place['Date'], df_place['target'])
plt.title(f'Target over time for Place_ID {place_id}')
plt.xlabel('Date')
plt.ylabel('Target')
plt.grid(True)
plt.show()
df_ap['Date'] = pd.to_datetime(df_ap['Date'])

weekday_pollution = (
    df_ap
    .assign(weekday=df_ap['Date'].dt.day_name())
    .groupby('weekday')['target']
    .mean()
)

weekday_order = [
    'Monday', 'Tuesday', 'Wednesday', 'Thursday',
    'Friday', 'Saturday', 'Sunday'
]

weekday_pollution = weekday_pollution.reindex(weekday_order)

plt.figure(figsize=(10, 5))
weekday_pollution.plot(kind='bar')
plt.title('Average Pollution by Weekday')
plt.xlabel('Weekday')
plt.ylabel('Average target')
plt.xticks(rotation=45)
plt.grid(axis='y')
plt.show()

## Missing Values and Feature Setup


In [ ]:
# Checking for missing values
missing = pd.DataFrame(df_ap.isnull().sum(), columns=["Amount"])
missing['Percentage'] = round((missing['Amount'] / df_ap.shape[0]) * 100, 2)
missing[missing['Amount'] != 0]

# Get columns where missing data is 80% or more
cols_high_missing = df_ap.columns[df_ap.isnull().mean() >= 0.8].tolist()
print(cols_high_missing)

# Drop leakage columns
leakage_cols = ['target_min', 'target_max', 'target_variance', 'target_count']

# Drop columns with 80%+ missing
high_missing_cols = df_ap.columns[df_ap.isnull().mean() >= 0.8].tolist()

# convert date columns to date value
df_ap['Date'] = pd.to_datetime(df_ap['Date'])
df_test['Date'] = pd.to_datetime(df_test['Date'])
date_start = df_ap['Date'].min()

for df in [df_ap, df_test]:
    df['day_number'] = (df['Date'] - date_start).dt.days + 1
    df['week'] = ((df['day_number'] - 1) // 7) + 1
    df['month'] = df['Date'].dt.month
    df['day_of_week'] = df['Date'].dt.dayofweek


## Location-Based Validation Split


In [ ]:
# Combine wind components into wind speed and drop the original u/v components.
u_wind_col = 'u_component_of_wind_10m_above_ground'
v_wind_col = 'v_component_of_wind_10m_above_ground'

for df in [df_ap, df_test]:
    df['wind_speed_10m_above_ground'] = np.sqrt(
        df[u_wind_col] ** 2 + df[v_wind_col] ** 2
    )

df_ap = df_ap.drop(columns=[u_wind_col, v_wind_col])
df_test = df_test.drop(columns=[u_wind_col, v_wind_col])

df_ap[['wind_speed_10m_above_ground']].head()


In [ ]:
# Get unique Place_IDs
places = df_ap['Place_ID'].unique()
print(f"Total unique places: {len(places)}")

# Split place IDs into train and validation sets
from sklearn.model_selection import train_test_split

places_train, places_val = train_test_split(places, test_size=0.2, random_state=42)

# Filter rows based on place membership
train_split = df_ap[df_ap['Place_ID'].isin(places_train)]
val_split = df_ap[df_ap['Place_ID'].isin(places_val)]

print(f"Train rows: {len(train_split)}, Places: {len(places_train)}")
print(f"Val rows:   {len(val_split)},  Places: {len(places_val)}")

removed_cols = ['Place_ID X Date', 'Date', 'Place_ID']
drop_cols = leakage_cols + high_missing_cols + removed_cols

X_tr = train_split.drop(columns=drop_cols + ['target'])
y_tr = train_split['target']

X_val = val_split.drop(columns=drop_cols + ['target'])
y_val = val_split['target']

print('X_tr shape:', X_tr.shape)
print('X_val shape:', X_val.shape)
print('y_tr shape:', y_tr.shape)
print('y_val shape:', y_val.shape)


## Training Data Checks


In [ ]:
# Cleaning the training data set first:
# Check for missing values:
missing = X_tr.isnull().mean().sort_values(ascending=False)
print(missing[missing > 0])

# Check for data types
print(X_tr.dtypes.value_counts())

# Check if any non-numeric columns slipped through
print(X_tr.select_dtypes('object').columns.tolist())

# We look for outliers
print(X_tr.describe().T[['min', 'max', 'mean', 'std']])

# Looking for duplicate rows in train data set
print(f"Duplicate rows: {X_tr.duplicated().sum()}")

y_tr.hist(bins=50)
plt.title("PM2.5 distribution in training split")
plt.xlabel("PM2.5")
plt.show()

print(y_tr.describe())


## Imputation and Outlier Handling


In [ ]:
# Clean the training data only first; apply the learned transformations to validation and test.
test_removed_cols = ['Place_ID X Date', 'Date', 'Place_ID']
X_test = df_test.drop(columns=high_missing_cols + test_removed_cols)
X_test = X_test.reindex(columns=X_tr.columns)

imputer = SimpleImputer(strategy='median')
X_tr_imputed = pd.DataFrame(imputer.fit_transform(X_tr), columns=X_tr.columns)
X_val_imputed = pd.DataFrame(imputer.transform(X_val), columns=X_tr.columns)
X_test_imputed = pd.DataFrame(imputer.transform(X_test), columns=X_tr.columns)

# Verify no missing values remain
print(X_tr_imputed.isnull().sum().sum())    # should be 0
print(X_val_imputed.isnull().sum().sum())   # should be 0
print(X_test_imputed.isnull().sum().sum())  # should be 0

# Identify strongly right-skewed positive features and log-transform them
skewness = X_tr_imputed.skew().sort_values(ascending=False)
skewness_threshold = 2

non_transform_cols = ['day_number', 'week', 'month', 'day_of_week']
angle_keywords = ['azimuth_angle', 'zenith_angle']

log_transform_cols = [
    col for col in skewness[skewness > skewness_threshold].index
    if col not in non_transform_cols
    and not any(keyword in col for keyword in angle_keywords)
    and (X_tr_imputed[col] >= 0).all()
]

X_tr_transformed = X_tr_imputed.copy()
X_val_transformed = X_val_imputed.copy()
X_test_transformed = X_test_imputed.copy()

for col in log_transform_cols:
    X_tr_transformed[col] = np.log1p(X_tr_transformed[col])
    X_val_transformed[col] = np.log1p(X_val_transformed[col])
    X_test_transformed[col] = np.log1p(X_test_transformed[col])

print(f"Log-transformed {len(log_transform_cols)} features with skewness > {skewness_threshold}:")
print(log_transform_cols)

# Clip only continuous non-angle, non-calendar features using train-set 1st and 99th percentiles
clip_cols = [
    col for col in X_tr_transformed.columns
    if col not in non_transform_cols
    and not any(keyword in col for keyword in angle_keywords)
]

lower = X_tr_transformed[clip_cols].quantile(0.01)
upper = X_tr_transformed[clip_cols].quantile(0.99)

X_tr_clipped = X_tr_transformed.copy()
X_val_clipped = X_val_transformed.copy()
X_test_clipped = X_test_transformed.copy()

X_tr_clipped[clip_cols] = X_tr_transformed[clip_cols].clip(lower=lower, upper=upper, axis=1)
X_val_clipped[clip_cols] = X_val_transformed[clip_cols].clip(lower=lower, upper=upper, axis=1)
X_test_clipped[clip_cols] = X_test_transformed[clip_cols].clip(lower=lower, upper=upper, axis=1)

# Count how many values were clipped per feature in the training data
clipped_counts_by_feature = (
    X_tr_transformed[clip_cols] != X_tr_clipped[clip_cols]
).sum().sort_values(ascending=False)
print(clipped_counts_by_feature[clipped_counts_by_feature > 0])

print('Total clipped values:', int(clipped_counts_by_feature.sum()))
print('Features with clipped values:', int((clipped_counts_by_feature > 0).sum()))


## Save Cleaned Data


In [ ]:
# Save cleaned train and validation data for model training.
# Keep Place_ID and Date as metadata for later time-based feature engineering.
train_metadata = train_split[['Place_ID', 'Date']].reset_index(drop=True)
val_metadata = val_split[['Place_ID', 'Date']].reset_index(drop=True)
test_metadata = df_test[['Place_ID X Date', 'Place_ID', 'Date']].reset_index(drop=True)

clean_train = pd.concat([train_metadata, X_tr_clipped.reset_index(drop=True)], axis=1)
clean_train['target'] = y_tr.reset_index(drop=True)

clean_val = pd.concat([val_metadata, X_val_clipped.reset_index(drop=True)], axis=1)
clean_val['target'] = y_val.reset_index(drop=True)

clean_test = pd.concat([test_metadata, X_test_clipped.reset_index(drop=True)], axis=1)

clean_train.to_csv('data/clean_train.csv', index=False)
clean_val.to_csv('data/clean_val.csv', index=False)
clean_test.to_csv('data/clean_test.csv', index=False)

print('Saved cleaned data:')
print('data/clean_train.csv', clean_train.shape)
print('data/clean_val.csv', clean_val.shape)
print('data/clean_test.csv', clean_test.shape)


## Save Cleaned Data With Rolling Features

This creates a separate cleaned dataset with past rolling averages per `Place_ID`. It uses only predictor columns, not past target values.

In [ ]:
# Create past rolling averages per place without using target history.
rolling_windows = [3]
calendar_cols = ['day_number', 'week', 'month', 'day_of_week']

best_features = pd.read_csv('data/best_features.csv')
rf_top_features = best_features['random_forest'].dropna().tolist()

# The original u/v wind components were replaced with total wind speed earlier.
wind_component_cols = [
    'u_component_of_wind_10m_above_ground',
    'v_component_of_wind_10m_above_ground',
]
rf_top_features = [
    'wind_speed_10m_above_ground' if col in wind_component_cols else col
    for col in rf_top_features
]
rf_top_features = list(dict.fromkeys(rf_top_features))

rolling_base_cols = [
    col for col in rf_top_features
    if col in train_split.columns
    and col not in leakage_cols + high_missing_cols + ['target'] + calendar_cols
]

missing_rf_features = [col for col in rf_top_features if col not in train_split.columns]
if missing_rf_features:
    print('RF features not found in train_split:', missing_rf_features)

print('RF features used for rolling:', len(rolling_base_cols))

def add_place_rolling_features(df, base_cols, windows):
    df = df.sort_values(['Place_ID', 'Date']).copy()

    for window in windows:
        for col in base_cols:
            df[f'{col}_place_roll_{window}'] = (
                df.groupby('Place_ID')[col]
                .transform(lambda s: s.shift(1).rolling(window, min_periods=1).mean())
            )

    return df

train_split_rolling = add_place_rolling_features(train_split, rolling_base_cols, rolling_windows)
val_split_rolling = add_place_rolling_features(val_split, rolling_base_cols, rolling_windows)

# Keep only the RF-selected base features and their rolling averages.
rolling_feature_cols = [
    f'{col}_place_roll_{window}'
    for window in rolling_windows
    for col in rolling_base_cols
]
selected_rolling_cols = rolling_base_cols + rolling_feature_cols

X_tr_rolling = train_split_rolling[selected_rolling_cols]
y_tr_rolling = train_split_rolling['target']
X_val_rolling = val_split_rolling[selected_rolling_cols]
y_val_rolling = val_split_rolling['target']

# Apply the same cleaning logic as the regular cleaned files.
rolling_imputer = SimpleImputer(strategy='median')
X_tr_rolling_imputed = pd.DataFrame(
    rolling_imputer.fit_transform(X_tr_rolling),
    columns=X_tr_rolling.columns,
)
X_val_rolling_imputed = pd.DataFrame(
    rolling_imputer.transform(X_val_rolling),
    columns=X_tr_rolling.columns,
)

rolling_skewness = X_tr_rolling_imputed.skew().sort_values(ascending=False)
rolling_non_transform_cols = calendar_cols

rolling_log_transform_cols = [
    col for col in rolling_skewness[rolling_skewness > skewness_threshold].index
    if col not in rolling_non_transform_cols
    and not any(keyword in col for keyword in angle_keywords)
    and (X_tr_rolling_imputed[col] >= 0).all()
]

X_tr_rolling_transformed = X_tr_rolling_imputed.copy()
X_val_rolling_transformed = X_val_rolling_imputed.copy()

for col in rolling_log_transform_cols:
    X_tr_rolling_transformed[col] = np.log1p(X_tr_rolling_transformed[col])
    X_val_rolling_transformed[col] = np.log1p(X_val_rolling_transformed[col])

rolling_clip_cols = [
    col for col in X_tr_rolling_transformed.columns
    if col not in rolling_non_transform_cols
    and not any(keyword in col for keyword in angle_keywords)
]

rolling_lower = X_tr_rolling_transformed[rolling_clip_cols].quantile(0.01)
rolling_upper = X_tr_rolling_transformed[rolling_clip_cols].quantile(0.99)

X_tr_rolling_clipped = X_tr_rolling_transformed.copy()
X_val_rolling_clipped = X_val_rolling_transformed.copy()

X_tr_rolling_clipped[rolling_clip_cols] = X_tr_rolling_transformed[rolling_clip_cols].clip(
    lower=rolling_lower,
    upper=rolling_upper,
    axis=1,
)
X_val_rolling_clipped[rolling_clip_cols] = X_val_rolling_transformed[rolling_clip_cols].clip(
    lower=rolling_lower,
    upper=rolling_upper,
    axis=1,
)

train_rolling_metadata = train_split_rolling[['Place_ID', 'Date']].reset_index(drop=True)
val_rolling_metadata = val_split_rolling[['Place_ID', 'Date']].reset_index(drop=True)

clean_train_rolling = pd.concat([train_rolling_metadata, X_tr_rolling_clipped.reset_index(drop=True)], axis=1)
clean_train_rolling['target'] = y_tr_rolling.reset_index(drop=True)

clean_val_rolling = pd.concat([val_rolling_metadata, X_val_rolling_clipped.reset_index(drop=True)], axis=1)
clean_val_rolling['target'] = y_val_rolling.reset_index(drop=True)

clean_train_rolling.to_csv('data/clean_train_rolling.csv', index=False)
clean_val_rolling.to_csv('data/clean_val_rolling.csv', index=False)

print('Rolling base features:', len(rolling_base_cols))
print('Saved cleaned rolling data:')
print('data/clean_train_rolling.csv', clean_train_rolling.shape)
print('data/clean_val_rolling.csv', clean_val_rolling.shape)
